<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Bonus (optionnel, avancé) — Autoencodeurs parcimonieux & signal émergent
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Rappel : entraîner un autoencodeur dense et voir pourquoi son goulot d'étranglement reste polysémantique<br>
    - Entraîner un autoencodeur parcimonieux (Top-K) sur les activations Evo2 pour obtenir des caractéristiques isolées<br>
    - Chercher une structure émergente sans aucune étiquette — par ex. une périodicité de 3 liée au cadre de lecture<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Cette piste n'est **pas obligatoire** pour le livrable principal — elle est là si votre
groupe termine en avance et veut voir ce que l'apprentissage de représentation non
supervisé peut découvrir, avec **zéro** étiquette, à l'intérieur d'un modèle de langage
génomique.

#### **Partie 1 — Rappel : l'autoencodeur classique**

Un autoencodeur apprend à compresser son entrée à travers un **goulot d'étranglement**
(bottleneck) étroit et à la reconstruire — ce goulot le force à ne garder que la structure
la plus utile. Voyons cela d'abord sur nos propres vecteurs de k-mers, car c'est rapide et
ne nécessite rien d'Evo2.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day5/assets/AE.png"/>

In [ ]:
import sys
sys.path.append("src")

import torch
import torch.nn as nn
from data import load_all
from featurize import kmer_matrix

train_df = load_all("../../data/supervised/processed", max_rows=None)["train"]
X = torch.tensor(kmer_matrix(train_df["sequence"].tolist(), k=4), dtype=torch.float32)

d_in = X.shape[1]
# TODO : construisez un autoencodeur dense avec un goulot d'étranglement à 16 dimensions
# (Linear(d_in, 16) -> ReLU -> Linear(16, d_in))
vanilla_ae = ...
optimizer = ...

for epoch in range(20):
    optimizer.zero_grad()
    # TODO : reconstruction + perte MSE entre x_hat et X
    x_hat = ...
    loss = ...
    loss.backward()
    optimizer.step()
print(f"final reconstruction loss: {loss.item():.5f}")

In [ ]:
# --- Préparation des données (une seule fois) -------------------------------
# Ces deux jeux de données viennent d'appels à l'API Evo2. S'ils sont déjà là, on ne
# fait rien ; sinon on lance l'extraction, à condition que NVIDIA_API_KEY soit défini.
import os, subprocess, sys
from pathlib import Path

DATA = Path("../2-data")
PROBE_NAME = "probe-genome"

def ensure(cible: Path, script: str, args: list, quoi: str):
    """Lance `script` dans 2-data/ si `cible` n'existe pas encore."""
    if cible.exists():
        print(f"OK   {quoi} — déjà présent ({cible})")
        return
    if not os.environ.get("NVIDIA_API_KEY"):
        raise RuntimeError(
            f"{quoi} manquant, et NVIDIA_API_KEY n'est pas défini.\n"
            "  export NVIDIA_API_KEY=nvapi-...   puis relancez cette cellule,\n"
            "  ou demandez les données pré-extraites à l'encadrant."
        )
    print(f"...  extraction de {quoi} — appels API, patientez")
    subprocess.run([sys.executable, script] + args, cwd=DATA, check=True)
    print(f"OK   {quoi} extrait")

# 1) activations non étiquetées, pour entraîner le SAE   (~17 min d'API)
ensure(DATA / "autoencoder" / "train" / "meta.json",
       "extract_autoencoder_activations.py",
       ["--raw_dir", "./raw/train", "--out_dir", "./autoencoder", "--split", "train",
        "--window", "4096", "--token_stride", "1", "--max_windows", "20"],
       "activations SAE (train)")

# 2) piste génomique CONTINUE + annotations, pour la sonde de la Partie 4
#    Un fichier à part, et un génome au GFF complet : c'est le seul jeu où
#    activations et annotations sont liées position par position.
ensure(DATA / "probe" / PROBE_NAME / "meta.json",
       "prepare_probe_genome.py",
       ["--fasta", "./raw/train/GCA_000240015.1_ASM24001v1.fasta",
        "--gff",   "./raw/train/GCA_000240015.1_ASM24001v1.gff",
        "--name", PROBE_NAME, "--out_dir", "./probe",
        "--length", "32768", "--chunk", "8192"],
       "piste génomique annotée (sonde)")

Ce goulot d'étranglement est **dense** : chacun des 16 nombres est utilisé
pour chaque entrée, et ils ne sont pas individuellement interprétables — une seule
« caractéristique » est généralement un mélange de nombreuses causes sous-jacentes (c'est
la *polysémanticité*).

#### **Partie 2 — Ce qu'apporte un autoencodeur parcimonieux**

Un autoencodeur **parcimonieux** (SAE) fait l'inverse : au lieu d'un goulot *étroit*, il
utilise un goulot **surcomplet** (plus d'unités cachées que d'entrées), mais force
seulement une poignée (`k`) d'entre elles à être actives pour une entrée donnée. L'idée
(issue de recherches récentes en interprétabilité mécanistique sur les modèles de
langage) : avec suffisamment de capacité et la bonne parcimonie, les caractéristiques
individuelles ont tendance à devenir **monosémantiques** — chacune suit un motif
distinct, ce qui les rend inspectables.

Ici, nous entraînons un SAE sur des **activations brutes d'Evo2** — des états internes
par position de nucléotide, extraits sans aucune information d'étiquette — et nous
cherchons des caractéristiques qui correspondent à quelque chose de réel, comme le cadre
de lecture d'un codon (structure de période 3).

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day5/assets/SAE.png"/>

In [ ]:
from embeddings import load_autoencoder_activations
from models.sae import TopKSparseAutoencoder, train_sae

acts = load_autoencoder_activations("../2-data/autoencoder", "train")
print("full activation dump shape:", acts.shape)

# on ne charge en mémoire qu'un sous-échantillon — c'est un memmap sur un fichier de 19 Go,
# ne le chargez pas en entier
N_SUBSAMPLE = 20_000
# TODO : convertissez acts[:N_SUBSAMPLE] en tensor float32
X_acts = ...
print("training on:", X_acts.shape)

In [ ]:
# --- Entraînement du SAE, avec sauvegarde ------------------------------------
# Le SAE est long à entraîner : on le sauvegarde pour pouvoir revenir directement
# sonder les caractéristiques (Partie 4) sans tout refaire.
# Mettez FORCE_RETRAIN = True pour le ré-entraîner malgré la sauvegarde.
from pathlib import Path

SAE_PATH = Path("../2-data/models/sae_evo2.pt")
FORCE_RETRAIN = False

D_HIDDEN, K, EPOCHS = X_acts.shape[1] * 8, 32, 30

if SAE_PATH.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(SAE_PATH)
    sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_hidden=ckpt["d_hidden"], k=ckpt["k"])
    sae.load_state_dict(ckpt["state_dict"])
    sae.eval()
    history = ckpt.get("history", {})
    print(f"SAE rechargé depuis {SAE_PATH}")
    print(f"  d_in={ckpt['d_in']} d_hidden={ckpt['d_hidden']} k={ckpt['k']} "
          f"| entraîné sur {ckpt.get('n_train', '?')} vecteurs")
else:
    # TODO : instanciez TopKSparseAutoencoder(d_in=X_acts.shape[1], d_hidden=D_HIDDEN, k=K)
    # puis entraînez-le avec train_sae(..., epochs=EPOCHS, batch_size=512)
    sae = ...
    sae, history = ...

    SAE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": sae.state_dict(),
                "d_in": X_acts.shape[1], "d_hidden": D_HIDDEN, "k": K,
                "n_train": int(X_acts.shape[0]), "history": history},
               SAE_PATH)
    print(f"SAE sauvegardé -> {SAE_PATH}")

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Partie 3 — La sonde : quelles caractéristiques discriminent gène / non-gène ?**

Jusqu'ici, tout s'est passé sans étiquette : le dump d'entraînement est un empilement de
fenêtres tirées au hasard, sans coordonnées, et on ne peut donc pas y rattacher un GFF.

Pour juger ce que le SAE a appris, il faut un **autre** jeu de données, préparé pour ça :
une **piste continue** d'un génome (`prepare_probe_genome.py`), où la ligne *i* des
activations correspond exactement au nucléotide *start + i*, avec son étiquette
codant / non-codant tirée du GFF.

Deux précautions y sont prises :

- la région est **contiguë** — les morceaux envoyés à l'API sont recollés dans l'ordre,
  jamais tirés au hasard ;
- le génome utilisé a un **GFF complet** (~87 % du génome couvert). Un GFF tronqué
  étiquetterait « non-codantes » des positions qui sont en fait des gènes, et toute la
  lecture serait fausse. Le script refuse de tourner en dessous de 50 % de couverture.

In [ ]:
from probe import GenomeProbe, SAEInterpreter, SignalScore

probe = GenomeProbe.load("../2-data/probe", PROBE_NAME)
annotations = probe.annotations()      # (début, fin, étiquette, couleur)
probe

In [ ]:
# activations du SAE, position par position
# TODO : feature_ts = SAEInterpreter.extract_features(sae, probe.activations)
feature_ts = ...

unique_features = SAEInterpreter.get_all_unique_active_features(feature_ts)
print(f"{len(unique_features)} caractéristiques actives sur {feature_ts.shape[1]}")

#### **Quelles caractéristiques séparent codant et non-codant ?**

Le **score de Fisher** compare, pour chaque caractéristique, l'écart de moyenne entre les
deux types de régions rapporté à sa variabilité : il récompense un écart **net et
stable**, pas un pic isolé.

`SignalScore` propose aussi `selectivity_score` (différence de fréquence d'allumage) et
`mutual_info` (plus lent, capte le non-linéaire).

In [ ]:
# TODO : scores = SignalScore.fisher_score(feature_ts, probe.coding)
scores = ...
# TODO : top_features = SignalScore.rank(scores, top=100)
top_features = ...

for j in top_features[:6]:
    c, n = feature_ts[probe.coding, j], feature_ts[~probe.coding, j]
    print(f"f{j:<6d} fisher={scores[j]:.3f} "
          f"| moyenne codant={c.mean():.3f} vs non-codant={n.mean():.3f}")

In [ ]:
SAEInterpreter.plot_features(
    feature_ts=feature_ts,
    selected_indices=top_features[:6],
    annotations=annotations,
    title="Activations du SAE + annotations | blocks.26",
    same_scale=False,
)

#### **L'autre signature : la périodicité de 3**

L'ADN codant est lu par triplets. Une caractéristique qui suit le cadre de lecture
oscille avec une **période 3** à l'intérieur des gènes et perd cette régularité en
dehors. `check_periodicity` mesure ce rapport signal/bruit à la fréquence 1/3.

In [ ]:
periodicite = SAEInterpreter.rank_by_periodicity(feature_ts,
                                                 candidates=unique_features[:3000])
for j, s in periodicite[:6]:
    print(f"f{j:<6d} rapport de puissance à la période 3 = {s:.1f}")

SAEInterpreter.plot_features(
    feature_ts=feature_ts,
    selected_indices=[j for j, _ in periodicite[:4]],
    annotations=annotations,
    title="Caractéristiques les plus périodiques (cadre de lecture)",
    same_scale=False,
)

#### **Comment lire ces figures**

Fond **orange** = région codante, fond **violet** = non codante. Chaque ligne a sa
propre échelle (`same_scale=False`), sinon les caractéristiques de faible amplitude
seraient écrasées.

Une caractéristique intéressante montre un contraste **consistant** : dense et forte
dans l'orange puis quasi éteinte dans le violet, ou l'inverse — les deux sens sont
également informatifs.

Deux réserves à formuler à voix haute :

1. On a **choisi** ces caractéristiques parmi des dizaines de milliers. Avec assez de
   candidats, on trouve toujours quelque chose : la validation, c'est de retrouver les
   **mêmes** caractéristiques sur une autre région (`probe.slice(...)`) ou un autre génome.
2. Le score de Fisher suppose des distributions à peu près régulières. Croisez-le avec
   `selectivity_score` avant de conclure.

#### **Discussion**

- Une caractéristique est-elle ressortie avec une forte périodicité de 3 ? Cela
  suggérerait que le modèle — sans étiquette, sans tâche, juste « reconstruire mes
  propres activations » — a représenté en interne le cadre de lecture des codons.
- Les caractéristiques les mieux classées par Fisher séparent-elles visiblement les
  régions orange des violettes ? Dans quel sens : présence dans les gènes, ou absence ?
- Refaites la sélection sur une **autre portion** de la piste (`probe.slice(a, b)`) :
  retrouve-t-on les mêmes caractéristiques ? C'est le seul vrai test — sur des dizaines
  de milliers de candidates, quelques-unes ressortent toujours par hasard.
- C'est exploratoire, et ce n'est pas noté sur l'obtention d'un « beau » résultat :
  l'important est de voir qu'une structure interprétable *peut* émerger d'un
  entraînement purement non supervisé.

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin de la semaine 4 &mdash; f&eacute;licitations !</div>
      <div>D&#39;un g&eacute;nome brut &agrave; un classifieur distill&eacute; : donn&eacute;es, mod&egrave;les de r&eacute;f&eacute;rence, embeddings Evo2,
      distillation, compression &mdash; et un aper&ccedil;u de l&#39;interpr&eacute;tabilit&eacute;. Gardez vos notebooks :
      ils sont la trace de ce que vous avez construit.</div>
      <div style="margin-top: 0.5em; font-size: 0.85em;">EEIA &middot; bioAI Workshop &mdash; Semaine 4</div>
    </div>
  </div>
</div>